<a href="https://colab.research.google.com/github/Gerardo-MauricioGP/Proyecto_SAM3/blob/main/Proyecto_SAM3_V3_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

#Install SAM 3 and extra dependencies

In [ ]:
import torch
import torchvision
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("Proyecto_SAM3")

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e ".[notebooks]"
%cd /content
!pip install -q supervision jupyter_bbox_widget

!pip uninstall -y cc_torch; TORCH_CUDA_ARCH_LIST="8.0 9.0"; pip install git+https://github.com/ronghanghu/cc_torch
!pip uninstall -y torch_generic_nms; TORCH_CUDA_ARCH_LIST="8.0 9.0"; pip install git+https://github.com/ronghanghu/torch_generic_nms

##Imports

In [ ]:
import cv2
import torch

import numpy as np
import supervision as sv

from pathlib import Path
from PIL import Image
from typing import Optional
from IPython.display import Video

from sam3.model_builder import build_sam3_video_predictor

HOME = Path.cwd()
print("HOME:", HOME)

###Load SAM3 model

In [ ]:
# use all available GPUs on the machine
# DEVICES = range(torch.cuda.device_count())

# # use only a single GPU
DEVICES = [torch.cuda.current_device()]

In [ ]:
predictor = build_sam3_video_predictor(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz", gpus_to_use=DEVICES)

#Video Descargado


In [ ]:
!gdown -q 1g8P4U-m82shswssj9sn2mXKYKBv8C8pM -O /content/0618.mp4

Optimizar calidad y FPS del video para ahorrar memoria RAM

Se ajustó la configuración del video a 10fps y se disminuyó la resolución/calidad. Esto soluciona los problemas de alto consumo de RAM en la ejecución.

In [ ]:
import os
import shutil

# Ruta en tu Drive
ruta_en_drive = "/content/drive/MyDrive/0618.mp4"
video_local = "/content/video_input.mp4"

# El nombre exacto del archivo que descargó la CELDA 1
video_descargado = "/content/0618.mp4"

if os.path.exists(ruta_en_drive):
    print(f" Copiando video desde Drive montado...")
    !cp "{ruta_en_drive}" "{video_local}"

    if os.path.exists(video_local):
        print(f"¡Éxito! Video listo desde tu Drive en: {video_local}")
    else:
        print(" Error crítico al copiar el archivo desde Drive.")

else:
    print(f" No se encontró el archivo en la ruta de Drive. ¡Activando Plan B!")

    # Buscamos el video descargado
    if os.path.exists(video_descargado):
        # Lo movemos y renombramos a video_input.mp4
        shutil.move(video_descargado, video_local)
        print(f" ¡Éxito! Video de respaldo en la nube preparado en: {video_local}")
    else:
        print("Error: No se encontró el video en Drive ni la descarga. ¡Corre la Celda 1 primero!")

In [ ]:
## Comprimir video con reducción agresiva de frames y resolución
VIDEO = "/content/video_input.mp4"
SOURCE_VIDEO_COMPRESSED = "/content/video_input_lowres.mp4"

if os.path.exists(VIDEO):
    print(f"DEBUG: Found input video at '{VIDEO}'. Proceeding with compression.")
    # Reducimos a 10 FPS y escala de 1080p para minimizar uso de RAM
    !ffmpeg -y -loglevel error -i "{VIDEO}" -vf "fps=10,scale=-2:1080" -vcodec libx264 -crf 28 "{SOURCE_VIDEO_COMPRESSED}"
    if os.path.exists(SOURCE_VIDEO_COMPRESSED):
        print(" Video ultra-optimizado exitosamente.")
    else:
        print(" Error: Compresión de video falló. El archivo de salida no se creó. Esto puede ocurrir si ffmpeg no tiene los códecs necesarios o hay un problema con el archivo de entrada.")
else:
    print(f" Error: El video de entrada '{VIDEO}' no se encontró. Asegúrate de ejecutar la celda de copia desde Drive (e7256740) antes de esta.")

In [ ]:
import os
from pathlib import Path

# Video comprimido para extraer los frames
SOURCE_VIDEO = Path("/content/video_input_lowres.mp4")
SOURCE_FRAMES = Path("/content/video_frames")
SOURCE_FRAMES.mkdir(parents=True, exist_ok=True)

if SOURCE_VIDEO.exists():
    print(f" Extrayendo frames optimizados de: {SOURCE_VIDEO}")
    !rm -rf {SOURCE_FRAMES}/*
    # Extraemos los frames del video que ya tiene 10fps
    !ffmpeg -i "{SOURCE_VIDEO}" -q:v 2 -start_number 0 "{SOURCE_FRAMES}/%05d.jpg"
    print(f" {len(os.listdir(SOURCE_FRAMES))} frames guardados en: {SOURCE_FRAMES}")
else:
    print(f" Error: No se encontró el video optimizado.")

In [ ]:
def load_frame(directory: str, index: int):
    """
    Loads a frame with a specific index from a directory where frames are named
    using the pattern '%05d.jpg' (e.g., 00000.jpg, 00001.jpg, 00002.jpg).

    Args:
        directory (str): Path to the directory containing image frames.
        index (int): Frame index (0-based).

    Returns:
        numpy.ndarray: Loaded frame in BGR format.

    Raises:
        FileNotFoundError: If the frame does not exist or cannot be read.
    """
    directory_path = Path(directory)
    frame_path = directory_path / f"{index:05d}.jpg"

    if not frame_path.exists():
        raise FileNotFoundError(f"Frame not found: {frame_path}")

    frame = cv2.imread(str(frame_path))
    if frame is None:
        raise FileNotFoundError(f"Failed to load frame: {frame_path}")

    return frame

### Inicio de la sesión de inferencia de video


SAM 3 requiere una inferencia con estado para la segmentación interactiva de video, por lo que debemos inicializar una sesión de inferencia en este video. Durante la inicialización, carga todos los fotogramas JPEG del directorio de video y almacena sus características en el estado de la sesión.

In [ ]:
import os
from pathlib import Path

# Usamos la versión optimizada para evitar que el servidor se reinicie
SOURCE_VIDEO = Path("/content/video_input_lowres.mp4")
print(f"DEBUG: Checking existence of SOURCE_VIDEO: {SOURCE_VIDEO.as_posix()}")

if SOURCE_VIDEO.exists():
    try:
        print(f" Iniciando sesión con video optimizado: {SOURCE_VIDEO.name}")
        response = predictor.handle_request(
            request=dict(
                type="start_session",
                resource_path=SOURCE_VIDEO.as_posix(),
            )
        )
        session_id = response["session_id"]
        print(f" Sesión iniciada exitosamente con ID: {session_id}")
    except Exception as e:
        print(f" Error al iniciar sesión: {e}")
        session_id = None
else:
    print(f" Error: No se encontró el video optimizado en {SOURCE_VIDEO.as_posix()}. Ejecuta la celda de compresión primero.")
    session_id = None

In [ ]:
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

### Liberar VRAM (memoria de la GPU)

Para intentar liberar memoria de la GPU y evitar errores de `OutOfMemoryError`, puedes ejecutar el siguiente comando. Es buena práctica hacerlo antes de operaciones que sabes que consumen mucha memoria.

In [ ]:
import torch

# Libera la caché de la GPU
torch.cuda.empty_cache()
print(' VRAM liberada.')


#Configuración de utilidades y visualización

Aquí definimos las funciones auxiliares para cargar los fotogramas del video y decodificar los resultados de SAM 3. También configuramos la paleta de colores y la función de anotación que dibujará automáticamente las máscaras de segmentación y las etiquetas sobre cada objeto rastreado.

In [ ]:
import cv2
import supervision as sv
from pathlib import Path

def load_frame(directory: str, index: int):
    directory_path = Path(directory)
    frame_path = directory_path / f"{index:05d}.jpg"
    if not frame_path.exists():
        raise FileNotFoundError(f"No se encontró el frame: {frame_path}")
    frame = cv2.imread(str(frame_path))
    return frame

try:
    frame_idx = 0
    # SOURCE_FRAMES se definió en la celda
    frame = load_frame(SOURCE_FRAMES, frame_idx)
    print(f" Frame {frame_idx} cargado correctamente.")
    sv.plot_image(frame)
except Exception as e:
    print(f" Error al cargar frame: {e}")
    print("Asegúrate de que la celda de extracción de frames (rve0HGsg4gfp) se ejecutó con éxito.")

def from_sam(result: dict) -> sv.Detections:
    return sv.Detections(
        xyxy=sv.mask_to_xyxy(result["out_binary_masks"]),
        mask=result["out_binary_masks"],
        confidence=result["out_probs"],
        tracker_id=result["out_obj_ids"],
    )

COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

from typing import Optional, List

def annotate(image: np.ndarray, detections: sv.Detections, labels: Optional[List[str]] = None) -> np.ndarray:
    h, w, _ = image.shape
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=(w, h))

    mask_annotator = sv.MaskAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.TRACK,
        opacity=0.6
    )

    annotated_image = image.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)

    label_text_content = []
    if labels and len(labels) == len(detections.tracker_id):
        # Si se proporcionan etiquetas y coinciden en número con los objetos detectados,
        # las usamos directamente.
        for i, tracker_id in enumerate(detections.tracker_id):
            label_text_content.append(f"#{tracker_id} {labels[i]}")
    else:
        # Si no hay etiquetas o no coinciden, mostramos solo el ID del rastreador.
        for tracker_id in detections.tracker_id:
            label_text_content.append(f"#{tracker_id}")

    if label_text_content:
        label_annotator = sv.LabelAnnotator(
            color=COLOR,
            color_lookup=sv.ColorLookup.TRACK,
            text_scale=text_scale,
            text_color=sv.Color.BLACK,
            text_position=sv.Position.TOP_CENTER,
            text_offset=(0, -30)
        )
        annotated_image = label_annotator.annotate(annotated_image, detections, label_text_content)

    return annotated_image

#Divide y Vencerás

###  Fase 1: Rastreo Aislado de Robots (Estrategia "Divide y Vencerás")

En esta fase definimos el motor de propagación de video y lo aplicamos **exclusivamente a los robots** para evitar colisiones en la memoria de SAM 3.

**Proceso:**
1. **Limpieza (`reset_session`):** Borramos la memoria de la IA para iniciar en blanco.
2. **Fijación del objetivo (`add_prompt`):** Instruimos al modelo buscar "robot" en el fotograma 0.
3. **Propagación:** Hacemos el *tracking* a lo largo de todo el video y guardamos el historial de forma aislada en la variable `frame_outputs_robots`.

In [ ]:
import gc
import torch

def propagate_in_video(predictor, session_id):
    frame_outputs = {}

    #  BLINDAJE DOBLE: Apagamos el historial (inference_mode) + Mitad de peso (autocast)
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        for response in predictor.handle_stream_request(
            request=dict(
                type="propagate_in_video",
                session_id=session_id,
            )
        ):
            idx = response["frame_index"]
            outputs = response["outputs"]

            cpu_outputs = {}
            for k, v in outputs.items():
                if torch.is_tensor(v):
                    #  .detach() rompe la cadena con la VRAM antes de mandarlo a la RAM
                    cpu_outputs[k] = v.detach().clone().cpu()
                else:
                    cpu_outputs[k] = v

            frame_outputs[idx] = cpu_outputs

            # Limpieza estructural
            del outputs

            # Purga cada 10 frames
            if idx > 0 and idx % 10 == 0:
                torch.cuda.empty_cache()
                gc.collect()
                print(f" Frame {idx} analizado. VRAM purgada obligatoriamente.")

    return frame_outputs

In [ ]:
# --- CELDA 1: RASTREO DE ROBOTS  ---
print(" INICIANDO RASTREO DE ROBOTS: BARRIDO 1 ...")

# 1. BARRIDO HACIA ADELANTE
predictor.handle_request(request=dict(type="reset_session", session_id=session_id))
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=0, text="robot")
    )
barrido_fwd_robots = propagate_in_video(predictor=predictor, session_id=session_id)


print(" INICIANDO RASTREO DE ROBOTS: BARRIDO 2 (FINAL AL 0)...")
# Ultimo Frame del video
ULTIMO_FRAME = 319

# 2. BARRIDO HACIA ATRÁS
predictor.handle_request(request=dict(type="reset_session", session_id=session_id))
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=ULTIMO_FRAME, text="robot")
    )
barrido_bwd_robots = propagate_in_video(predictor=predictor, session_id=session_id)



# 3. LA LÓGICA DE RELLENO

print(" INICIANDO FUSIÓN Y RELLENO DE ROBOTS PERDIDOS...")
frame_outputs_robots = {}

# Juntamos los números de todos los frames procesados
todos_los_frames_r = set(list(barrido_fwd_robots.keys()) + list(barrido_bwd_robots.keys()))

def mascara_valida_robot(out):
    if out is None: return False
    # Si la lista de IDs de robots detectados está vacía, se perdió
    if len(out.get("out_obj_ids", [])) == 0: return False
    return True

# Revisamos frame por frame
for idx in sorted(todos_los_frames_r):
    out_fwd_r = barrido_fwd_robots.get(idx)
    out_bwd_r = barrido_bwd_robots.get(idx)

    # Si el Barrido 1 encontró a los robots, lo usamos
    if mascara_valida_robot(out_fwd_r):
        frame_outputs_robots[idx] = out_fwd_r

    # Si el Barrido 1 falló , pero el Barrido 2 SÍ los trae, rellenamos
    elif mascara_valida_robot(out_bwd_r):
        frame_outputs_robots[idx] = out_bwd_r
        print(f" Rescate exitoso: Robots en frame {idx} recuperados con el barrido 2.")

    # Si de plano no salen en ninguno, guardamos vacío
    else:
        frame_outputs_robots[idx] = out_fwd_r if out_fwd_r is not None else out_bwd_r

print(" ¡Fusión completada! Variable 'frame_outputs_robots' lista y sellada para el marcador.")

###  Fase 2: Rastreo Aislado del Balón

Continuando con la estrategia, ahora procesamos la pelota de forma independiente. El paso más crítico aquí es borrar la memoria previa para evitar que SAM colapse al intentar mantener los IDs de los robots y del balón al mismo tiempo.

**Proceso:**
1. **Amnesia Estratégica (`reset_session`):** Borramos el registro de los robots de la memoria de la IA.
2. **Nuevo Objetivo (`add_prompt`):** Instruimos al modelo buscar "ball" en el fotograma 0.
3. **Almacenamiento Aislado:** Hacemos el *tracking* y guardamos este nuevo recorrido en una libreta distinta llamada `frame_outputs_ball`.

In [ ]:
# --- CELDA 2: RASTREO DEL BALÓN ---
print(" INICIANDO RASTREO DEL BALÓN...")

predictor.handle_request(request=dict(type="reset_session", session_id=session_id))

#  APLICAMOS PRECISIÓN MIXTA AL PROMPT INICIAL
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=0, text="ball")
    )

frame_outputs_ball = propagate_in_video(predictor=predictor, session_id=session_id)
print(f" Recorrido del balón guardado en memoria.")

### Fase 3: Rastreo Aislado de la Portería Amarilla

Continuando con la estrategia 'Divide y Vencerás', ahora procesamos la portería amarilla de forma independiente. Esto asegura que SAM 3 enfoque sus recursos únicamente en este objeto sin interferir con los rastreos previos de robots y balón.

**Proceso:**
1.  **Reiniciar Sesión (`reset_session`):** Borramos el registro de cualquier objeto anterior de la memoria de la IA.
2.  **Nuevo Objetivo (`add_prompt`):** Instruimos al modelo buscar 'yellow rectangle' en el fotograma 0.
3.  **Almacenamiento Aislado:** Realizamos el seguimiento y guardamos este nuevo recorrido en una variable separada llamada `frame_outputs_yellow_rectangle`.

In [ ]:
# --- CELDA 3: RASTREO DE LA PORTERÍA AMARILLA ---
print(" INICIANDO RASTREO DE LA PORTERÍA AMARILLA...")

predictor.handle_request(request=dict(type="reset_session", session_id=session_id))

# Aplicamos precisión mixta al prompt inicial
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=0, text="yellow rectangle")
    )

frame_outputs_yellow_rectangle = propagate_in_video(predictor=predictor, session_id=session_id)
print(f" Recorrido de la portería amarilla guardado en memoria.")

Ahora que tenemos los datos de seguimiento de la portería amarilla, actualizaremos la función `callback` en la celda `4jnW0CEgAbwW` para incluir esta información, asegurando que la máscara se mueva con el objeto en el video final.

### Fase 3: Rastreo Aislado del campo

Continuando con la estrategia 'Divide y Vencerás', ahora procesamos la portería amarilla de forma independiente. Esto asegura que SAM 3 enfoque sus recursos únicamente en este objeto sin interferir con los rastreos previos de robots y balón.

**Proceso:**
1.  **Reiniciar Sesión (`reset_session`):** Borramos el registro de cualquier objeto anterior de la memoria de la IA.
2.  **Nuevo Objetivo (`add_prompt`):** Instruimos al modelo buscar 'yellow rectangle' en el fotograma 0.
3.  **Almacenamiento Aislado:** Realizamos el seguimiento y guardamos este nuevo recorrido en una variable separada llamada `frame_outputs_yellow_rectangle`.

In [ ]:
# --- CELDA 3: RASTREO DEL CAMPO (DOBLE BARRIDO INTELIGENTE) ---
print(" INICIANDO RASTREO: BARRIDO 1 (0 AL FINAL)...")

# 1. BARRIDO HACIA ADELANTE
predictor.handle_request(request=dict(type="reset_session", session_id=session_id))
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=0, text="green playing field")
    )
barrido_fwd = propagate_in_video(predictor=predictor, session_id=session_id)


print(" INICIANDO RASTREO: BARRIDO 2 (FINAL AL 0)...")
# Ajusta este número si tu video tiene más o menos frames (ej. 290 o 300)
ULTIMO_FRAME = 319

# 2. BARRIDO HACIA ATRÁS
predictor.handle_request(request=dict(type="reset_session", session_id=session_id))
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=ULTIMO_FRAME, text="green playing field")
    )
barrido_bwd = propagate_in_video(predictor=predictor, session_id=session_id)



# 3. LA LÓGICA DE RELLENO

print(" INICIANDO FUSIÓN Y RELLENO DE MÁSCARAS VACÍAS...")
frame_outputs_field = {}

# Juntamos los números de todos los frames procesados
todos_los_frames = set(list(barrido_fwd.keys()) + list(barrido_bwd.keys()))

# Función rápida para saber si la IA nos devolvió una máscara vacía
def mascara_valida(out):
    if out is None: return False
    # Si la lista de IDs detectados está vacía, es que perdió el rastro
    if len(out.get("out_obj_ids", [])) == 0: return False
    return True

# Revisamos frame por frame
for idx in sorted(todos_los_frames):
    out_fwd = barrido_fwd.get(idx)
    out_bwd = barrido_bwd.get(idx)

    # Si el Barrido 1 encontró la cancha, lo usamos
    if mascara_valida(out_fwd):
        frame_outputs_field[idx] = out_fwd

    # Si el Barrido 1 falló o está vacío, pero el Barrido 2 SÍ la encontró, rellenamos el hueco
    elif mascara_valida(out_bwd):
        frame_outputs_field[idx] = out_bwd
        print(f" Rescate exitoso: Frame {idx} rellenado con el barrido 2.")

    # Si los dos fallaron, guardamos el resultado vacío para que el código no truene
    else:
        frame_outputs_field[idx] = out_fwd if out_fwd is not None else out_bwd

print(" ¡Fusión completada! Variable 'frame_outputs_field' lista y sellada.")

### Fase 4: Rastreo Aislado de la Portería Azul

Para completar nuestra estrategia 'Divide y Vencerás', ahora procesaremos la portería azul de forma independiente. Esto garantiza que SAM 3 asigne recursos de seguimiento específicamente para este objeto.

**Proceso:**
1.  **Reiniciar Sesión (`reset_session`):** Borramos el registro de cualquier objeto anterior de la memoria de la IA.
2.  **Nuevo Objetivo (`add_prompt`):** Instruimos al modelo buscar 'blue rectangle' en el fotograma 0.
3.  **Almacenamiento Aislado:** Realizamos el seguimiento y guardamos este nuevo recorrido en una variable separada llamada `frame_outputs_blue_rectangle`.

In [ ]:
# --- CELDA 4: RASTREO DE LA PORTERÍA AZUL ---
print(" INICIANDO RASTREO DE LA PORTERÍA AZUL...")

predictor.handle_request(request=dict(type="reset_session", session_id=session_id))

# Aplicamos precisión mixta al prompt inicial
with torch.autocast(device_type="cuda", dtype=torch.float16):
    predictor.handle_request(
        request=dict(type="add_prompt", session_id=session_id, frame_index=0, text="blue panel")
    )

frame_outputs_blue_rectangle = propagate_in_video(predictor=predictor, session_id=session_id)
print(f" Recorrido de la portería azul guardado en memoria.")

#Motor de renderizado y lógica del marcador
Aquí definimos la función principal callback que construye el video fotograma por fotograma. Esta función superpone las máscaras de segmentación de la cancha, las porterías, los robots y el balón. Además, incluye la lógica espacial de colisiones: detecta cuándo el balón intersecta con las porterías y es tocado por los robots para actualizar y dibujar automáticamente el marcador en la pantalla.



In [ ]:
import math
import cv2
import numpy as np
import supervision as sv

#Ejecutar
TARGET_VIDEO = HOME / f"{SOURCE_VIDEO.stem}-tracked.mov"
TARGET_VIDEO_COMPRESSED = HOME / f"{TARGET_VIDEO.stem}-web.mp4"


# VARIABLES GLOBALES DEL MARCADOR

red_team_score = 0
blue_team_score = 0
ball_in_yellow_rectangle_prev_frame = False
ball_in_blue_rectangle_prev_frame = False

dilation_pixels = 3

# Paletas de colores
ROBOT_CUSTOM_COLORS = sv.ColorPalette.from_hex([
    "#FF0000", # Red for tracker_id 0
    "#0000FF", # Blue for tracker_id 1
    "#FF0000"  # Red for tracker_id 2
])

default_mask_annotator = sv.MaskAnnotator(color=COLOR, color_lookup=sv.ColorLookup.TRACK)
default_label_annotator = sv.LabelAnnotator(
    color=COLOR, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK
)

robot_mask_annotator = sv.MaskAnnotator(color=ROBOT_CUSTOM_COLORS, color_lookup=sv.ColorLookup.TRACK)
robot_label_annotator = sv.LabelAnnotator(
    color=ROBOT_CUSTOM_COLORS, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK
)

def callback(frame: np.ndarray, index: int) -> np.ndarray:
    global red_team_score, blue_team_score
    global ball_in_yellow_rectangle_prev_frame, ball_in_blue_rectangle_prev_frame
    global dilation_pixels

    annotated_frame = frame.copy()

    det_robots = None
    det_ball = None
    det_yellow_rectangle = None
    det_blue_rectangle = None

    # --- PASO 0: DIBUJAR LA CANCHA (FONDO) ---
    if index in frame_outputs_field:
        out_f = frame_outputs_field[index]
        det_field = from_sam(out_f)
        if len(det_field) > 0:
            det_field.tracker_id = np.array([10] * len(det_field.tracker_id)) # Color verde de tu paleta COLOR
            labels_f = ["Cancha"] * len(det_field.tracker_id)
            annotated_frame = default_mask_annotator.annotate(scene=annotated_frame, detections=det_field)
            annotated_frame = default_label_annotator.annotate(scene=annotated_frame, detections=det_field, labels=labels_f)

    # --- PASO A: DIBUJAR PORTERÍAS ---
    if index in frame_outputs_yellow_rectangle:
        out_yr = frame_outputs_yellow_rectangle[index]
        det_yellow_rectangle = from_sam(out_yr)
        if len(det_yellow_rectangle) > 0:
            det_yellow_rectangle.tracker_id = np.array([0] * len(det_yellow_rectangle)) # Color amarillo
            labels_yr = ["Porteria Amarilla"] * len(det_yellow_rectangle)
            annotated_frame = default_mask_annotator.annotate(scene=annotated_frame, detections=det_yellow_rectangle)
            annotated_frame = default_label_annotator.annotate(scene=annotated_frame, detections=det_yellow_rectangle, labels=labels_yr)

    # (NUEVO) DIBUJAR PORTERÍA AZUL
    if index in frame_outputs_blue_rectangle:
        out_br = frame_outputs_blue_rectangle[index]
        det_blue_rectangle = from_sam(out_br)
        if len(det_blue_rectangle) > 0:
            det_blue_rectangle.tracker_id = np.array([7] * len(det_blue_rectangle)) # Color azul de tu paleta
            labels_br = ["Porteria Azul"] * len(det_blue_rectangle)
            annotated_frame = default_mask_annotator.annotate(scene=annotated_frame, detections=det_blue_rectangle)
            annotated_frame = default_label_annotator.annotate(scene=annotated_frame, detections=det_blue_rectangle, labels=labels_br)

    # --- PASO B: DIBUJAR LOS ROBOTS ---
    if index in frame_outputs_robots:
        out_r = frame_outputs_robots[index]
        det_robots = from_sam(out_r)
        if len(det_robots) > 0:
            labels_r = [f"Robot #{t_id}" for t_id in det_robots.tracker_id]
            annotated_frame = robot_mask_annotator.annotate(scene=annotated_frame, detections=det_robots)
            annotated_frame = robot_label_annotator.annotate(scene=annotated_frame, detections=det_robots, labels=labels_r)

    # --- PASO C: DIBUJAR EL BALÓN ---
    if index in frame_outputs_ball:
        out_b = frame_outputs_ball[index]
        det_ball = from_sam(out_b)
        if len(det_ball) > 0:
            det_ball.tracker_id = np.array([20] * len(det_ball.tracker_id))
            labels_b = ["Ball"] * len(det_ball.tracker_id)
            annotated_frame = default_mask_annotator.annotate(scene=annotated_frame, detections=det_ball)
            annotated_frame = default_label_annotator.annotate(scene=annotated_frame, detections=det_ball, labels=labels_b)


    # LÓGICA DE GOL Y COLISIONES

    ball_in_yellow_current = False
    ball_in_blue_current = False
    red_touching_ball = False
    blue_touching_ball = False

    if det_ball is not None and len(det_ball.mask) > 0:
        kernel = np.ones((dilation_pixels, dilation_pixels), np.uint8)
        dilated_ball_masks = np.array([cv2.dilate(m.astype(np.uint8), kernel, iterations=1) for m in det_ball.mask])

        # Revisar si el balón entra en la portería amarilla
        if det_yellow_rectangle is not None and len(det_yellow_rectangle.mask) > 0:
            if np.any(sv.mask_iou_batch(dilated_ball_masks, det_yellow_rectangle.mask) > 0):
                ball_in_yellow_current = True

        # Revisar si el balón entra en la portería azul
        if det_blue_rectangle is not None and len(det_blue_rectangle.mask) > 0:
            if np.any(sv.mask_iou_batch(dilated_ball_masks, det_blue_rectangle.mask) > 0):
                ball_in_blue_current = True

        # Revisar toques de robots
        if det_robots is not None and len(det_robots.mask) > 0:
            # Equipo Rojo (IDs 0 y 2)
            red_indices = [i for i, tid in enumerate(det_robots.tracker_id) if tid in [0, 2]]
            if len(red_indices) > 0 and np.any(sv.mask_iou_batch(det_robots.mask[red_indices], det_ball.mask) > 0):
                red_touching_ball = True

            # Equipo Azul (ID 1)
            blue_indices = [i for i, tid in enumerate(det_robots.tracker_id) if tid == 1]
            if len(blue_indices) > 0 and np.any(sv.mask_iou_batch(det_robots.mask[blue_indices], det_ball.mask) > 0):
                blue_touching_ball = True

    # --- CONDICIONES DE MARCADOR ---
    # Asumiendo que el Rojo anota en la portería Amarilla
    if red_touching_ball and ball_in_yellow_current and not ball_in_yellow_rectangle_prev_frame:
        red_team_score += 1
        print(f"¡GOL ROJO! Marcador: Rojo {red_team_score} - Azul {blue_team_score} (Frame {index})")

    # Asumiendo que el Azul anota en la portería Azul (puedes cambiarlo si es al revés)
    if blue_touching_ball and ball_in_blue_current and not ball_in_blue_rectangle_prev_frame:
        blue_team_score += 1
        print(f"¡GOL AZUL! Marcador: Rojo {red_team_score} - Azul {blue_team_score} (Frame {index})")

    ball_in_yellow_rectangle_prev_frame = ball_in_yellow_current
    ball_in_blue_rectangle_prev_frame = ball_in_blue_current

    # Dibujar Marcador en pantalla
    cv2.putText(annotated_frame, f"Rojos: {red_team_score} | Azules: {blue_team_score}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 4, cv2.LINE_AA)
    cv2.putText(annotated_frame, f"Rojos: {red_team_score} | Azules: {blue_team_score}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2, cv2.LINE_AA)

    return annotated_frame

print(" Generando video final uniendo todo el entorno...")
sv.process_video(
    source_path=SOURCE_VIDEO,
    target_path=TARGET_VIDEO,
    callback=callback
)

print("⚙️ Optimizando para visualización web...")
!ffmpeg -y -loglevel error -i "{TARGET_VIDEO}" -c:v libx264 -pix_fmt yuv420p -profile:v main -level 3.1 -movflags +faststart "{TARGET_VIDEO_COMPRESSED}"
print(f" Video listo: {TARGET_VIDEO_COMPRESSED}")

#Procesamiento y exportación de video
Esta sección ejecuta el procesamiento del video aplicando la función de renderizado. Al terminar, utiliza ffmpeg para comprimir el archivo resultante en un formato .mp4 ligero y compatible con la web, y genera un reproductor interactivo directamente dentro de Colab para que puedas visualizar y descargar tu render final.

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

if os.path.exists(str(TARGET_VIDEO_COMPRESSED)):
    video_file = open(str(TARGET_VIDEO_COMPRESSED), 'rb').read()
    video_url = 'data:video/mp4;base64,' + b64encode(video_file).decode()

    display(HTML(f'''
    <h3>Video Resultante</h3>
    <video width="640" height="480" controls>
          <source src="{video_url}" type="video/mp4">
          Tu navegador no soporta el video.
    </video>
    <br>
    <a href="{video_url}" download="resultado_seguimiento.mp4" style="color: #007bff; font-weight: bold;">Click aquí para descargar el video si no carga</a>
    '''))
else:
    print(" El video procesado no se encontró. Asegúrate de ejecutar la celda anterior.")